## **Random Forest STL na deskryptorach**

Wykorzystana reprezentacja: **10-cechowe deskryptory 2D**

**Lista endpointów**:

1. Caco-2 (Wang)
2. Lipophilicity (AstraZeneca)
3. Solubility (AqSolDB)
4. HIA (Hou)
5. Half Life (Obach)
6. Clearance Hepatocyte (AstraZeneca)
7. CYP3A4 Inhibition (Veith)
8. VDss (Lombardo)
9. AMES Mutagenicity
10. hERG (Wang)

**Metryki jakości**: AUROC (klasyfikacja), RMSE(regresja), Accuracy, F1 (klasyfikacja), R^2(regresja), MAE(regresja)

Krok 1: Przygotowanie danych i zamiana na deskryptory

In [1]:
!pip install rdkit pandas numpy scikit-learn -U
!pip install pytdc --no-dependencies
!pip install fuzzywuzzy

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytdc 1.1.15 requires cellxgene-census==1.15.0, which is not installed.
pytdc 1.1.15 requires dataclasses<1.0,>=0.6, which is not installed.
pytdc 1.1.15 requires evaluate==0.4.2, which is not installed.
pytdc 1.1.15 requires gget<1.0.0,>=0.28.4, which is not installed.
pytdc 1.1.15 requires tiledbsoma<2.0.0,>=1.7.2, which is not installed.
pytdc 1.1.15 requires accelerate==0.33.0, but you have accelerate 1.13.0 which is incompatible.
pytdc 1.1.15 requires datasets<2.20.0, but you have datasets 4.0.0 which is incompatible.
pytdc 1.1.15 requires huggingface-hub<1.0,>=0.20.3, but you have huggingface-hub 1.10.1 which is incompatible.
pytdc 1.1.15 requires numpy<2.0.0,>=1.26.4, but you have numpy 2.4.4 which is incompatible.
pytdc 1.1.15 requires pandas<3.0.0,>=2.1.4, but you have pandas 3.0.2 which is incompatible.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
data_folder = "/content/drive/MyDrive/MLDD - ADMET/data_splits" #folder z zapisanymi zestawami danych aby umożliwić porównanie danych na dokładnie tych samych zestawach dla każdego modelu

In [4]:
def load_split_pickle(dataset_name):
    filepath = f"{data_folder}/{dataset_name}_split.pkl"
    with open(filepath, "rb") as f:
        split = pickle.load(f)
    return split["train"], split["test"]

In [5]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, MolSurf, AllChem
from sklearn.preprocessing import StandardScaler

class ADMETDescriptorFeaturizer:
    def __init__(self, y_column, smiles_col='Drug'):
        self.y_column = y_column
        self.smiles_col = smiles_col
        # Lista nazw deskryptorów, które wyliczamy
        self.feature_names = [
            'MW', 'LogP', 'HBD', 'HBA', 'TPSA',
            'RotatableBonds', 'AromaticRings', 'HeavyAtoms',
            'MolMR', 'FractionCSP3'
        ]

        self.scaler = StandardScaler()

    def __call__(self, df, fit_scaler=True):
        features = []
        labels = []

        for i, row in df.iterrows():
            smiles = row[self.smiles_col]
            y = row[self.y_column]
            mol = Chem.MolFromSmiles(smiles)

            if mol:
                desc_vector = [
                    # 1-4: Reguła Lipinskiego
                    Descriptors.MolWt(mol),          # Masa cząsteczkowa
                    Descriptors.MolLogP(mol),        # Lipofilowość
                    Lipinski.NumHDonors(mol),        # Donory H
                    Lipinski.NumHAcceptors(mol),      # Akceptor H

                    # 5-6: Reguły Vebera
                    Descriptors.TPSA(mol),           # Powierzchnia polarna
                    Lipinski.NumRotatableBonds(mol), # Elastyczność

                    # 7-8: Model ESOL / Rozmiar
                    Lipinski.NumAromaticRings(mol),  # Pierścienie aromatyczne
                    Descriptors.HeavyAtomCount(mol), # Liczba atomów ciężkich

                    # 9-10: Polaryzowalność i Trójwymiarowość
                    Descriptors.MolMR(mol),          # Refrakcja molowa
                    Lipinski.FractionCSP3(mol)       # Stopień nasycenia (3D)
                ]
                features.append(desc_vector)
                labels.append(y)
            else:
                print(f"Błąd w SMILES na indeksie {i}: {smiles}")

        features_array = np.array(features)

        # Ważne: Deskryptory mają różne skale (MW to setki, HBD to jednostki)
        # Wykonujemy normalizację (Z-score scaling)
        if fit_scaler:
            # Uczymy skaler na danych treningowych i transformujemy je
            normalized_features = self.scaler.fit_transform(features_array)
        else:
            # Używamy wyuczonego skalera na danych testowych (unikamy data leakage)
            normalized_features = self.scaler.transform(features_array)

        return normalized_features, np.array(labels)

Funkcje pomocnicze do zapisywania i wczytywania danych oraz wypisywania metryk

In [6]:
import pickle

def split_and_save(tdc_data, dataset_name):
    split    = tdc_data.get_split(seed=42, frac=[0.8, 0.0, 0.2])
    train_df = split["train"]
    test_df  = split["test"]

    filepath = f"{dataset_name}_split.pkl"
    with open(filepath, "wb") as f:
        pickle.dump({"train": train_df, "test": test_df}, f)

    return train_df, test_df

In [7]:
def print_metrics(metrics, task='classification'):
    print(f"\n{'='*40}")
    print(f"Best params: {metrics['best_params']}")
    print(f"{'='*40}")
    if task == 'classification':
        print(f"  Accuracy : {metrics['test_metrics']['accuracy']:.4f}")
        print(f"  F1       : {metrics['test_metrics']['f1']:.4f}")
        print(f"  AUROC    : {metrics['test_metrics']['auroc']:.4f}")
    else:
        print(f"  RMSE     : {metrics['test_metrics']['rmse']:.4f}")
        print(f"  MAE      : {metrics['test_metrics']['mae']:.4f}")
        print(f"  R²       : {metrics['test_metrics']['r2']:.4f}")
    print(f"{'='*40}\n")


def save_metrics(metrics, dataset_name, filepath, task='classification'):
    with open(filepath, 'a') as f:
        f.write(f"\n{'='*40}\n")
        f.write(f"Endpoint    : {dataset_name}\n")
        f.write(f"{'='*40}\n")
        f.write(f"Best params: {metrics['best_params']}\n")
        f.write(f"{'-'*40}\n")
        if task == 'classification':
            f.write(f"  Accuracy : {metrics['test_metrics']['accuracy']:.4f}\n")
            f.write(f"  F1       : {metrics['test_metrics']['f1']:.4f}\n")
            f.write(f"  AUROC    : {metrics['test_metrics']['auroc']:.4f}\n")
        else:
            f.write(f"  RMSE     : {metrics['test_metrics']['rmse']:.4f}\n")
            f.write(f"  MAE      : {metrics['test_metrics']['mae']:.4f}\n")
            f.write(f"  R²       : {metrics['test_metrics']['r2']:.4f}\n")
        f.write(f"{'='*40}\n")

Krok 2: Trening Random Forest

In [8]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def train_rf_regression(X_train, X_test, y_train, y_test, rf_param_grid=None, cv_folds=5):

    if rf_param_grid is None:
        rf_param_grid = {
            "n_estimators": [200, 500, 1000, 1500],
            "max_depth": [None, 10,15,20,25,30],
            "min_samples_split": [2, 5,10],
            "max_features": ["sqrt", "log2"]

        }

    n_iter = 1
    for values in rf_param_grid.values():
        n_iter *= len(values)

    random_search = RandomizedSearchCV(
        estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
        param_distributions=rf_param_grid,
        cv=cv_folds,
        n_iter=n_iter,
        scoring='neg_root_mean_squared_error',
        n_jobs=-1,
        random_state=42
    )
    random_search.fit(X_train, y_train)
    best_params = random_search.best_params_

    final_model = random_search.best_estimator_

    y_test_pred = final_model.predict(X_test)

    metrics = {
        "best_params": best_params,
        "test_metrics": {
            "rmse": np.sqrt(mean_squared_error(y_test, y_test_pred)),
            "mae":  mean_absolute_error(y_test, y_test_pred),
            "r2":   r2_score(y_test, y_test_pred)
        },
        "model": final_model
    }

    return metrics

In [9]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

def train_rf_classification(X_train, X_test, y_train, y_test, rf_param_grid=None, cv_folds=5):

    if rf_param_grid is None:
        rf_param_grid = {
          "n_estimators": [200, 500, 1000, 1500],
            "max_depth": [None, 10,15,20,25,30],
            "min_samples_split": [2, 5,10],
            "max_features": ["sqrt", "log2"]
        }

    n_iter = 1
    for values in rf_param_grid.values():
        n_iter *= len(values)

    random_search = RandomizedSearchCV(
        estimator=RandomForestClassifier(random_state=42, n_jobs=-1, class_weight="balanced"),
        param_distributions=rf_param_grid,
        cv=cv_folds,
        n_iter=n_iter,
        scoring='roc_auc',
        n_jobs=-1,
        random_state=42
    )
    random_search.fit(X_train, y_train)
    best_params = random_search.best_params_

    final_model = random_search.best_estimator_

    y_test_pred      = final_model.predict(X_test)
    y_test_pred_prob = final_model.predict_proba(X_test)[:, 1]

    metrics = {
        "best_params": best_params,
        "test_metrics": {
            "accuracy": accuracy_score(y_test, y_test_pred),
            "f1":       f1_score(y_test, y_test_pred),
            "auroc":    roc_auc_score(y_test, y_test_pred_prob),
        },
        "model": final_model
    }

    return metrics

Endpoint 1: Caco-2 (Wang)

In [ ]:
from tdc.single_pred import ADME

data = ADME(name = 'Caco2_Wang')

train, test = load_split_pickle('Caco2_Wang')

featurizer = ADMETDescriptorFeaturizer(y_column="Y")

X_train, y_train = featurizer(train)
X_test,  y_test  = featurizer(test)

metrics = train_rf_regression(X_train, X_test, y_train, y_test)

print_metrics(metrics, task="regression")
save_metrics(metrics, "Caco2_Wang", "metrics_ADMET_featurizer.txt", task="regression")


Found local copy...
Loading...
Done!



Best params: {'n_estimators': 200, 'min_samples_split': 2, 'max_features': 'sqrt', 'max_depth': 20}
  RMSE     : 0.4643
  MAE      : 0.3450
  R²       : 0.6606



Endpoint2: Lipophilicity

In [ ]:
from tdc.single_pred import ADME

data = ADME(name = 'Lipophilicity_AstraZeneca')

train, test = load_split_pickle('Lipophilicity_AstraZeneca')

featurizer = ADMETDescriptorFeaturizer(y_column="Y")

X_train, y_train = featurizer(train)
X_test,  y_test  = featurizer(test)

metrics = train_rf_regression(X_train, X_test, y_train, y_test)

print_metrics(metrics, task="regression")
save_metrics(metrics, 'Lipophilicity_AstraZeneca', "metrics_ADMET_featurizer.txt", task="regression")

Downloading...
100%|██████████| 298k/298k [00:00<00:00, 1.17MiB/s]
Loading...
Done!



Best params: {'n_estimators': 1500, 'min_samples_split': 2, 'max_features': 'sqrt', 'max_depth': None}
  RMSE     : 0.8819
  MAE      : 0.6778
  R²       : 0.4736



Endpoint 3: Solubility

In [ ]:
from tdc.single_pred import ADME

data = ADME(name = 'Solubility_AqSolDB')

train, test = load_split_pickle('Solubility_AqSolDB')

featurizer = ADMETDescriptorFeaturizer(y_column="Y")

X_train, y_train = featurizer(train)
X_test,  y_test  = featurizer(test)

metrics = train_rf_regression(X_train, X_test, y_train, y_test)

print_metrics(metrics, task="regression")
save_metrics(metrics, 'Solubility_AqSolDB', "metrics_ADMET_featurizer.txt", task="regression")

Downloading...
100%|██████████| 853k/853k [00:00<00:00, 2.83MiB/s]
Loading...
Done!
[11:59:29] WARNING: not removing hydrogen atom without neighbors
[11:59:30] WARNING: not removing hydrogen atom without neighbors
[11:59:30] WARNING: not removing hydrogen atom without neighbors
[11:59:30] WARNING: not removing hydrogen atom without neighbors
[11:59:30] WARNING: not removing hydrogen atom without neighbors
[11:59:30] WARNING: not removing hydrogen atom without neighbors
[11:59:30] WARNING: not removing hydrogen atom without neighbors
[11:59:30] WARNING: not removing hydrogen atom without neighbors
[11:59:30] WARNING: not removing hydrogen atom without neighbors
[11:59:30] WARNING: not removing hydrogen atom without neighbors
[11:59:30] WARNING: not removing hydrogen atom without neighbors
[11:59:30] WARNING: not removing hydrogen atom without neighbors
[11:59:30] WARNING: not removing hydrogen atom without neighbors
[11:59:30] WARNING: not removing hydrogen atom without neighbors
[11:59

Błąd w SMILES na indeksie 3814: CC1=CC=C[NH+2]([O-])[CH-]1
Błąd w SMILES na indeksie 4005: O=C(O)C1=C[NH+2]([O-])[CH-]C=C1


[11:59:32] WARNING: not removing hydrogen atom without neighbors
[11:59:32] WARNING: not removing hydrogen atom without neighbors
[11:59:33] WARNING: not removing hydrogen atom without neighbors
[11:59:33] WARNING: not removing hydrogen atom without neighbors
[11:59:35] WARNING: not removing hydrogen atom without neighbors
[11:59:35] WARNING: not removing hydrogen atom without neighbors
[11:59:35] WARNING: not removing hydrogen atom without neighbors
[11:59:35] WARNING: not removing hydrogen atom without neighbors
[11:59:35] WARNING: not removing hydrogen atom without neighbors
[11:59:35] WARNING: not removing hydrogen atom without neighbors
[11:59:35] WARNING: not removing hydrogen atom without neighbors
[11:59:36] WARNING: not removing hydrogen atom without neighbors
[11:59:36] WARNING: not removing hydrogen atom without neighbors
[11:59:36] WARNING: not removing hydrogen atom without neighbors
[11:59:36] WARNING: not removing hydrogen atom without neighbors
[11:59:36] WARNING: not r

KeyboardInterrupt: 

Endpoint 4: HIA

In [ ]:
from tdc.single_pred import ADME

data = ADME(name = 'HIA_Hou')

train, test = load_split_pickle('HIA_Hou')

featurizer = ADMETDescriptorFeaturizer(y_column="Y")

X_train, y_train = featurizer(train)
X_test,  y_test  = featurizer(test)

metrics = train_rf_classification(X_train, X_test, y_train, y_test)

print_metrics(metrics)
save_metrics(metrics, 'HIA_Hou', "metrics_ADMET_featurizer.txt")

Downloading...
100%|██████████| 40.1k/40.1k [00:00<00:00, 648kiB/s]
Loading...
Done!



Best params: {'n_estimators': 1000, 'min_samples_split': 5, 'max_features': 'sqrt', 'max_depth': None}
  Accuracy : 0.9138
  F1       : 0.9510
  AUROC    : 0.8812



Endpoint 5: Half Life

In [ ]:
from tdc.single_pred import ADME

data = ADME(name = 'Half_Life_Obach')

train, test = load_split_pickle('Half_Life_Obach')

featurizer = ADMETDescriptorFeaturizer(y_column="Y")

X_train, y_train = featurizer(train)
X_test,  y_test  = featurizer(test)

metrics = train_rf_regression(X_train, X_test, y_train, y_test)

print_metrics(metrics, task="regression")
save_metrics(metrics, 'Half_Life_Obach', "metrics_ADMET_featurizer.txt", task="regression")

Downloading...
100%|██████████| 53.6k/53.6k [00:00<00:00, 870kiB/s]
Loading...
Done!



Best params: {'n_estimators': 500, 'min_samples_split': 2, 'max_features': 'sqrt', 'max_depth': 10}
  RMSE     : 112.4331
  MAE      : 26.9898
  R²       : 0.1187



Endpoint 6: Clearance Hepatocyte

In [ ]:
from tdc.single_pred import ADME

data = ADME(name = 'Clearance_Hepatocyte_AZ')

train, test = load_split_pickle('Clearance_Hepatocyte_AZ')

featurizer = ADMETDescriptorFeaturizer(y_column="Y")

X_train, y_train = featurizer(train)
X_test,  y_test  = featurizer(test)

metrics = train_rf_regression(X_train, X_test, y_train, y_test)

print_metrics(metrics, task="regression")
save_metrics(metrics, "Clearance_Hepatocyte_AZ", "metrics_ADMET_featurizer.txt", task="regression")

Downloading...
100%|██████████| 91.6k/91.6k [00:00<00:00, 738kiB/s]
Loading...
Done!



Best params: {'n_estimators': 1500, 'min_samples_split': 10, 'max_features': 'sqrt', 'max_depth': None}
  RMSE     : 47.1139
  MAE      : 36.9005
  R²       : 0.1108



Endpoint 7: CYP3A4 Inhibition

In [ ]:
from tdc.single_pred import ADME

data = ADME(name = 'CYP3A4_Veith')

train, test = load_split_pickle('CYP3A4_Veith')

featurizer = ADMETDescriptorFeaturizer(y_column="Y")

X_train, y_train = featurizer(train)
X_test,  y_test  = featurizer(test)

metrics = train_rf_classification(X_train, X_test, y_train, y_test)

print_metrics(metrics)
save_metrics(metrics, 'CYP3A4_Veith', "metrics_ADMET_featurizer.txt")

Downloading...
100%|██████████| 746k/746k [00:00<00:00, 2.20MiB/s]
Loading...
Done!


KeyboardInterrupt: 

Endpoint 8: VDss

In [ ]:
from tdc.single_pred import ADME

data = ADME(name = 'VDss_Lombardo')

train, test = load_split_pickle('VDss_Lombardo')

featurizer = ADMETDescriptorFeaturizer(y_column="Y")

X_train, y_train = featurizer(train)
X_test,  y_test  = featurizer(test)

metrics = train_rf_regression(X_train, X_test, y_train, y_test)

print_metrics(metrics, task="regression")
save_metrics(metrics, 'VDss_Lombardo', "metrics_ADMET_featurizer.txt", task="regression")

Endpoint 9: AMES Mutagenicity

In [ ]:
from tdc.single_pred import Tox

data = Tox(name = 'AMES')

train, test = load_split_pickle('AMES')

featurizer = ADMETDescriptorFeaturizer(y_column="Y")

X_train, y_train = featurizer(train)
X_test,  y_test  = featurizer(test)

metrics = train_rf_classification(X_train, X_test, y_train, y_test)

print_metrics(metrics)
save_metrics(metrics, 'AMES', "metrics_ADMET_featurizer.txt")

Downloading...
100%|██████████| 344k/344k [00:00<00:00, 514kiB/s]
Loading...
Done!


Endpoint 10: hERG (Wang)



In [ ]:
from tdc.single_pred import Tox

data = Tox(name = 'hERG')

train, test = load_split_pickle('hERG')

featurizer = ADMETDescriptorFeaturizer(y_column="Y")

X_train, y_train = featurizer(train)
X_test,  y_test  = featurizer(test)

metrics = train_rf_classification(X_train, X_test, y_train, y_test)

print_metrics(metrics)
save_metrics(metrics, 'hERG', "metrics_ADMET_featurizer.txt")

Found local copy...
Loading...
Done!
[08:40:12] WARNING: not removing hydrogen atom without neighbors
[08:40:12] WARNING: not removing hydrogen atom without neighbors



Best params: {'n_estimators': 200, 'min_samples_split': 2, 'max_features': 'sqrt', 'max_depth': 10}
  Accuracy : 0.8092
  F1       : 0.8731
  AUROC    : 0.8295

